# 03 Rule Recovery Track

Track B searches directly for the hidden XOR tap set. It consumes Track A outputs as priors, applies clean-prefix filtering, and ranks candidates with noise-aware log-likelihood and training accuracy.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

SEARCH_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(path for path in SEARCH_ROOTS if (path / "DAY2_data.txt").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sparsetap_config import DEFAULT_CONFIG
from sparsetap_utils import (
    build_rule_recovery_pool,
    candidate_table,
    evaluate_rule_candidate,
    load_candidates,
    prepare_environment,
    rank_candidates,
    run_beam_search,
    run_greedy_search,
    run_local_search,
    run_prefix_solver,
    run_reduced_exhaustive_search,
    save_candidates,
    save_table,
)

config = DEFAULT_CONFIG
data, prefix = prepare_environment(config)
predictive_candidates = load_candidates(config.candidate_dir / "predictive_candidates.json")

candidate_pool, support = build_rule_recovery_pool(predictive_candidates[: config.candidate_top_k], [], config)
prefix_only = [evaluate_rule_candidate(cand, data, prefix, config.noise_prob) for cand in run_prefix_solver(prefix, candidate_pool[:12], config)]
greedy = run_greedy_search(candidate_pool[:16], data, prefix, config)
beam = run_beam_search(candidate_pool[:16], data, prefix, config)
local_runs = []
for restart_idx, seed_candidate in enumerate(beam["candidates"][: config.local_search_restarts], start=1):
    local_result = run_local_search(seed_candidate["taps"], candidate_pool[:16], data, prefix, config)
    best_local = dict(local_result["best"])
    best_local["track"] = "rule_recovery"
    best_local["metadata"] = {**best_local.get("metadata", {}), "restart": restart_idx, "note": "Local search refinement from beam seed."}
    local_runs.append(best_local)
reduced = run_reduced_exhaustive_search(candidate_pool[:10], data, prefix, config, max_combination_size=4)

rule_candidates = rank_candidates(
    prefix_only
    + beam["candidates"][: config.candidate_top_k]
    + greedy["history"][: config.candidate_top_k]
    + local_runs
    + reduced[: config.candidate_top_k],
    data=data,
    prefix=prefix,
    noise=config.noise_prob,
)

save_candidates(rule_candidates, config.candidate_dir / "rule_recovery_candidates.json")
rule_table = candidate_table(rule_candidates)
save_table(rule_table, config.metrics_dir / "rule_recovery_summary.csv")
print("Candidate pool:", candidate_pool[:16])
rule_table.head(30)


The clean 64-bit prefix acts as a hard compatibility test here. That makes it easy later to explain why some high-likelihood noisy candidates still fail to be viable final rules.